In [1]:
import pandas as pd
import numpy as np


In [5]:
df = pd.read_csv('~/Downloads/archive (6)/winemag-data_first150k.csv')
df2 = pd.read_csv('~/Downloads/archive (6)/winemag-data-130k-v2.csv')
df3 = pd.read_json('~/Downloads/archive (6)/winemag-data-130k-v2.json')


In [6]:
from itertools import combinations

datasets = {
    "150k CSV": df,
    "130k CSV": df2,
    "JSON": df3
}

for name, data in datasets.items():
    print(f"{name}: {data.shape[0]:,} rows, {data.shape[1]} columns")
    print(f"Columns: {data.columns.tolist()}\n")

assert all("description" in data.columns for data in datasets.values())

review_sets = {}

for name, data in datasets.items():
    descriptions = data["description"].dropna().astype(str).str.strip().str.casefold()
    descriptions = descriptions[descriptions.ne("")]

    review_sets[name] = set(descriptions)

    print(f"{name}")
    print(f"  Unique descriptions: {descriptions.nunique():,}")
    print(f"  Duplicate descriptions: {descriptions.duplicated().sum():,}\n")

print("OVERLAP BETWEEN DATASETS")

for a, b in combinations(datasets, 2):
    overlap = len(review_sets[a] & review_sets[b])
    print(f"{a} vs {b}: {overlap:,} shared descriptions")

print("\nCOMBINED DATASET")

combined = pd.concat(
    [df, df3, df2],
    ignore_index=True
)

combined = combined.loc[:, ~combined.columns.str.startswith("Unnamed:")].copy()

combined["description_key"] = (
    combined["description"].astype("string").str.strip().str.casefold()
)

combined = combined.dropna(subset=["description_key"])
combined = combined[combined["description_key"].ne("")]

conflicts = (
    combined.groupby("description_key")["variety"]
    .nunique(dropna=True)
    .gt(1)
    .sum()
)

combined_clean = (
    combined.drop_duplicates(subset="description_key", keep="last")
    .drop(columns="description_key")
    .reset_index(drop=True)
)

print(f"Total reviews before deduplication: {len(combined):,}")
print(f"Unique reviews after deduplication: {len(combined_clean):,}")
print(f"Duplicates removed: {len(combined) - len(combined_clean):,}")
print(f"Descriptions with conflicting varieties: {conflicts:,}")

150k CSV: 150,930 rows, 11 columns
Columns: ['Unnamed: 0', 'country', 'description', 'designation', 'points', 'price', 'province', 'region_1', 'region_2', 'variety', 'winery']

130k CSV: 129,971 rows, 14 columns
Columns: ['Unnamed: 0', 'country', 'description', 'designation', 'points', 'price', 'province', 'region_1', 'region_2', 'taster_name', 'taster_twitter_handle', 'title', 'variety', 'winery']

JSON: 129,971 rows, 13 columns
Columns: ['points', 'title', 'description', 'taster_name', 'taster_twitter_handle', 'price', 'designation', 'variety', 'region_1', 'region_2', 'province', 'country', 'winery']

150k CSV
  Unique descriptions: 97,821
  Duplicate descriptions: 53,109

130k CSV
  Unique descriptions: 119,955
  Duplicate descriptions: 10,016

JSON
  Unique descriptions: 119,955
  Duplicate descriptions: 10,016

OVERLAP BETWEEN DATASETS
150k CSV vs 130k CSV: 48,346 shared descriptions
150k CSV vs JSON: 48,346 shared descriptions
130k CSV vs JSON: 119,955 shared descriptions

COMBIN

In [7]:
combined = pd.concat([df, df2], ignore_index=True)

combined = combined.drop(columns="Unnamed: 0", errors="ignore")
combined = combined.dropna(subset=["description", "variety"]).copy()

combined["description_key"] = (
    combined["description"].str.strip().str.casefold()
)

combined = combined[combined["description_key"].ne("")]

conflicts = (
    combined.groupby("description_key")["variety"]
    .nunique()
)

conflicting_keys = conflicts[conflicts > 1].index

combined_clean = (
    combined[~combined["description_key"].isin(conflicting_keys)]
    .drop_duplicates(subset="description_key", keep="last")
    .drop(columns="description_key")
    .reset_index(drop=True)
)

print("Original rows:", len(df) + len(df2))
print("Final rows:", len(combined_clean))
print("Conflicting descriptions excluded:", len(conflicting_keys))
print("Final shape:", combined_clean.shape)

combined_clean.head()

Original rows: 280901
Final rows: 169407
Conflicting descriptions excluded: 22
Final shape: (169407, 13)


,country,description,designation,points,price,province,region_1,region_2,variety,winery,taster_name,taster_twitter_handle,title
0,US,This tremendous 100% varietal wine hails from ...,Martha's Vineyard,96,235.0,California,Napa Valley,Napa,Cabernet Sauvignon,Heitz,NaN,NaN,NaN
1,Spain,"Ripe aromas of fig, blackberry and cassis are ...",Carodorum Selección Especial Reserva,96,110.0,Northern Spain,Toro,NaN,Tinta de Toro,Bodega Carmen Rodríguez,NaN,NaN,NaN
2,US,Mac Watson honors the memory of a wine once ma...,Special Selected Late Harvest,96,90.0,California,Knights Valley,Sonoma,Sauvignon Blanc,Macauley,NaN,NaN,NaN
3,US,"This spent 20 months in 30% new French oak, an...",Reserve,96,65.0,Oregon,Willamette Valley,Willamette Valley,Pinot Noir,Ponzi,NaN,NaN,NaN
4,France,"This is the top wine from La Bégude, named aft...",La Brûlade,95,66.0,Provence,Bandol,NaN,Provence red blend,Domaine de la Bégude,NaN,NaN,NaN


In [11]:
examples = duplicate_summary[
    duplicate_summary["unique_wineries"] > 1
]["description_key"].head(3)

with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.max_rows", None,
    "display.width", None
):
    for i, desc in enumerate(examples, 1):
        print(f"\n{'=' * 30} EXAMPLE {i} {'=' * 30}")

        display(
            duplicates[duplicates["description_key"] == desc]
            .drop(columns="description_key")
            .reset_index(drop=True)
        )


============================== EXAMPLE 1 ==============================


,country,description,designation,points,price,province,region_1,region_2,variety,winery,source,taster_name,taster_twitter_handle,title
0,France,"90–92. Barrel sample. Solid and structured, while also a fruity wine, this has weight, richness and generous black-fruit flavors. It's quite a success, and showcases the pleasurable side of the vintage.",Barrel Sample,91,NaN,Bordeaux,Haut-Médoc,NaN,Bordeaux-style Red Blend,Château Hanteillan,150k CSV,NaN,NaN,NaN
1,France,"90–92. Barrel sample. Solid and structured, while also a fruity wine, this has weight, richness and generous black-fruit flavors. It's quite a success, and showcases the pleasurable side of the vintage.",Barrel Sample,91,NaN,Bordeaux,Haut-Médoc,NaN,Bordeaux-style Red Blend,Château d'Hanteillan,130k CSV,Roger Voss,@vossroger,Château d'Hanteillan 2012 Barrel Sample (Haut-Médoc)



============================== EXAMPLE 2 ==============================


,country,description,designation,points,price,province,region_1,region_2,variety,winery,source,taster_name,taster_twitter_handle,title
0,France,"90–92. Barrel sample. This is a solid and dense wine with hints of bitter chocolate along with super-ripe blackberry fruits. The wine has concentration, richness and a fine ageing potential.",Barrel Sample,91,NaN,Bordeaux,Moulis-en-Médoc,NaN,Bordeaux-style Red Blend,Château Maucaillou,150k CSV,NaN,NaN,NaN
1,France,"90–92. Barrel sample. This is a solid and dense wine with hints of bitter chocolate along with super-ripe blackberry fruits. The wine has concentration, richness and a fine ageing potential.",Barrel Sample,91,NaN,Bordeaux,Margaux,NaN,Bordeaux-style Red Blend,Château Pontac-Lynch,150k CSV,NaN,NaN,NaN
2,France,"90–92. Barrel sample. This is a solid and dense wine with hints of bitter chocolate along with super-ripe blackberry fruits. The wine has concentration, richness and a fine ageing potential.",Barrel Sample,91,NaN,Bordeaux,Margaux,NaN,Bordeaux-style Red Blend,Château Pontac-Lynch,130k CSV,Roger Voss,@vossroger,Château Pontac-Lynch 2012 Barrel Sample (Margaux)
3,France,"90–92. Barrel sample. This is a solid and dense wine with hints of bitter chocolate along with super-ripe blackberry fruits. The wine has concentration, richness and a fine ageing potential.",Barrel Sample,91,NaN,Bordeaux,Moulis-en-Médoc,NaN,Bordeaux-style Red Blend,Château Maucaillou,130k CSV,Roger Voss,@vossroger,Château Maucaillou 2012 Barrel Sample (Moulis-en-Médoc)



============================== EXAMPLE 3 ==============================


,country,description,designation,points,price,province,region_1,region_2,variety,winery,source,taster_name,taster_twitter_handle,title
0,France,"92-94 Barrel sample. A big, concentrated, powerful wine, yet also one restrained by elegant fresh fruits. There is plenty of style here, a complex mix of fruits, acidity and balanced tannins.",Barrel sample,93,NaN,Bordeaux,Saint-Julien,NaN,Bordeaux-style Red Blend,Château Talbot,150k CSV,NaN,NaN,NaN
1,France,"92-94 Barrel sample. A big, concentrated, powerful wine, yet also one restrained by elegant fresh fruits. There is plenty of style here, a complex mix of fruits, acidity and balanced tannins.",Barrel sample,93,NaN,Bordeaux,Saint-Julien,NaN,Bordeaux-style Red Blend,Château Langoa Barton,150k CSV,NaN,NaN,NaN
2,France,"92-94 Barrel sample. A big, concentrated, powerful wine, yet also one restrained by elegant fresh fruits. There is plenty of style here, a complex mix of fruits, acidity and balanced tannins.",Barrel sample,93,NaN,Bordeaux,Saint-Julien,NaN,Bordeaux-style Red Blend,Château Langoa Barton,130k CSV,Roger Voss,@vossroger,Château Langoa Barton 2008 Barrel sample (Saint-Julien)
